In [1]:
# ⚠️ MODELO 1 — VERSIÓN ORIGINAL CON FUGA DE INFORMACIÓN
# Se conserva tal como se ejecutó (92.90% en Test) para documentar el error en el informe.
# La corrección está en los Notebooks 05, 06 y 07 (Modelo 2).
# Sus datos y su modelo se guardan aparte (data/processed/modelo1 y models/v1_con_fuga) para no mezclarse con el Modelo 2.

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Herramientas de evaluación de Machine Learning
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("📖 Cargando los conjuntos procesados desde data/processed/...")

train_path = "../data/processed/modelo1/train.parquet"
test_path = "../data/processed/modelo1/test.parquet"

df_train = pd.read_parquet(train_path)
df_test = pd.read_parquet(test_path)

print(f"✅ Datos cargados en memoria:")
print(f" • Conjunto de Entrenamiento: {len(df_train):,} decisiones")
print(f" • Conjunto de Prueba (Test):  {len(df_test):,} decisiones")

# Separar las variables de entrada (X) y la etiqueta objetivo (y)
features = ['calle', 'bote', 'to_call', 'stack', 'pot_odds']
target = 'accion'

X_train = df_train[features]
y_train = df_train[target]

X_test = df_test[features]
y_test = df_test[target]

print(f"\n🎯 Variables de entrada (X): {features}")
print(f"🎯 Variable a predecir   (y): '{target}' (0=Fold, 1=Check/Call, 2=Bet/Raise)")

📖 Cargando los conjuntos procesados desde data/processed/...
✅ Datos cargados en memoria:
 • Conjunto de Entrenamiento: 263,225 decisiones
 • Conjunto de Prueba (Test):  56,362 decisiones

🎯 Variables de entrada (X): ['calle', 'bote', 'to_call', 'stack', 'pot_odds']
🎯 Variable a predecir   (y): 'accion' (0=Fold, 1=Check/Call, 2=Bet/Raise)


In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

# 1. Baseline 1: Clase Mayoritaria (Adivinar siempre Fold a ciegas)
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
y_pred_b1 = dummy.predict(X_test)

acc_b1 = accuracy_score(y_test, y_pred_b1)
f1_b1 = f1_score(y_test, y_pred_b1, average='macro', zero_division=0)

# 2. Baseline 2: Árbol de Reglas Simples (Profundidad 3)
arbol_simple = DecisionTreeClassifier(max_depth=3, random_state=42)
arbol_simple.fit(X_train, y_train)
y_pred_b2 = arbol_simple.predict(X_test)

acc_b2 = accuracy_score(y_test, y_pred_b2)
f1_b2 = f1_score(y_test, y_pred_b2, average='macro', zero_division=0)

# 3. Construir la Tabla Comparativa Oficial Corregida
tabla_baselines = pd.DataFrame([
    {
        "Modelo / Enfoque": "Baseline 1: Clase Mayoritaria (Fold)",
        "Accuracy": f"{acc_b1 * 100:.2f}%",
        "F1-Score Macro": f"{f1_b1:.3f}",
        "Propósito Académico": "Piso mínimo absoluto (adivinar sin mirar contexto)"
    },
    {
        "Modelo / Enfoque": "Baseline 2: Árbol de Reglas Simples (Prof. 3)",
        "Accuracy": f"{acc_b2 * 100:.2f}%",
        "F1-Score Macro": f"{f1_b2:.3f}",
        "Propósito Académico": "Sentido común básico: pasar si es gratis, retirarse si cobran"
    },
    {
        "Modelo / Enfoque": "🎯 Objetivo Red Neuronal MLP (Paso 4)",
        "Accuracy": "78.00% - 80.00%",
        "F1-Score Macro": "> 0.720",
        "Propósito Académico": "Aprender el 28% de decisiones complejas (agresiones, pot-commitment, contexto no lineal)"
    }
])

print("=" * 95)
print("  TABLA COMPARATIVA DE REFERENCIA (BASELINES) — Doc. Maestro § 9.1 (Actualizada)")
print("=" * 95)
print(tabla_baselines.to_string(index=False))
print("=" * 95)

  TABLA COMPARATIVA DE REFERENCIA (BASELINES) — Doc. Maestro § 9.1 (Actualizada)
                             Modelo / Enfoque        Accuracy F1-Score Macro                                                                      Propósito Académico
         Baseline 1: Clase Mayoritaria (Fold)          48.10%          0.217                                       Piso mínimo absoluto (adivinar sin mirar contexto)
Baseline 2: Árbol de Reglas Simples (Prof. 3)          72.21%          0.662                            Sentido común básico: pasar si es gratis, retirarse si cobran
         🎯 Objetivo Red Neuronal MLP (Paso 4) 78.00% - 80.00%        > 0.720 Aprender el 28% de decisiones complejas (agresiones, pot-commitment, contexto no lineal)


### 🧠 Análisis Crítico de los Baselines y Justificación de la Meta (78% - 80%)

1. **El 72.21% del Baseline 2:** Un árbol superficial de 3 preguntas alcanza 72.21% porque en el póker las decisiones rutinarias son abrumadoras (si la apuesta es cero, casi el 100% pasa; si enfrentan una subida preflop sin cartas premium, la mayoría se retira).
2. **El verdadero valor de la Red Neuronal (El 28% restante):** La inteligencia artificial no se justifica para aprender lo obvio, sino para descifrar el **28% de jugadas complejas** donde el árbol falla:
   - Situaciones de jugador comprometido con el bote (*pot-committed*).
   - Momentos de agresión (*bet/raise*) condicionados por la relación stack/bote y posición.
   - Variaciones de probabilidad continua (*pot odds*) en función de la calle (*flop* vs *river*).
3. **Meta Científica:** Si la red neuronal logra capturar apenas una tercera parte de ese 28% de situaciones difíciles, la precisión global se elevará al rango de **78% - 80%**, alcanzando la frontera teórica esperable en un juego de información imperfecta con cartas privadas ocultas.